# Manual Price Push

Push manual price overrides for specific SKUs using market data or fixed prices.

## How to use
1. Run cells 1-3 (setup + data loading)
2. Edit the `PUSH_LIST` in cell 4 with your product IDs and desired actions
3. Run cell 4 — prices are computed per product × cohort (using cohort-specific market data)
4. Review the output table
5. Set `MODE = 'live'` and run cell 5 to push

> Each product is automatically expanded to all 9 cohorts. Market-based actions use cohort-specific prices.  
> Main/general cohorts (695, 61, 699, 697, 698, 696) are auto-mirrored by the push handler.

## Available Actions
| Action | Description |
|--------|-------------|
| `market_min` | Lowest market price (P0 / minimum) |
| `market_25` | 25th percentile market price |
| `market_50` | Median market price (P50) |
| `market_75` | 75th percentile market price |
| `market_max` | Highest market price (maximum) |
| `market_avg` | Average of min and max market prices |
| `target_margin` | Price from brand-category target margin |
| `step_up` | Next tier above current price in the price list |
| `step_down` | Next tier below current price in the price list |
| `<number>` | Fixed price (e.g. `115`, `23.5`) |

> Only SKUs with stock > 0 are pushed.

In [7]:
# =============================================================================
# SETUP
# =============================================================================
import sys, os
import pandas as pd
import numpy as np
import pytz
from datetime import datetime

sys.path.insert(0, os.path.abspath('.'))
import setup_environment_2
setup_environment_2.initialize_env()

from db import query_snowflake
from constants import WAREHOUSE_MAPPING, COHORT_IDS

os.chdir(os.path.join(os.path.dirname(os.path.abspath('__file__')) or os.getcwd(), 'modules'))
%run push_prices_handler.ipynb
os.chdir('..')

CAIRO_TZ = pytz.timezone('Africa/Cairo')
TODAY = datetime.now(CAIRO_TZ).strftime('%Y-%m-%d')
print(f'Ready. Today: {TODAY}')

/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json
Push Prices Handler loaded at 2026-04-21 12:38:10 Cairo time
✓ API credentials loaded successfully
✓ Google Sheets client initialized
Ready. Today: 2026-04-21


In [8]:
# =============================================================================
# LOAD MARKET DATA + WAC + CURRENT PRICES + PACKING UNITS
# =============================================================================
os.chdir('modules')
%run market_data_module_2.ipynb
os.chdir('..')

print('Loading market data (V2)...')
market_data = get_market_data_v2()
print(f'  Market data: {len(market_data)} rows')

print('Loading WAC...')
WAC_QUERY = f'''
SELECT product_id, wac_p
FROM finance.all_cogs c
WHERE wac_p > 0
    AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) 
        BETWEEN c.from_date AND c.to_date
'''
df_wac = query_snowflake(WAC_QUERY)
print(f'  WAC: {len(df_wac)} rows')

print('Loading current prices...')
PRICES_QUERY = f'''

    SELECT 
        cohort_id,
        pu.product_id,
        pu.packing_unit_id,
        pu.basic_unit_count,
        AVG(cpu.price) AS current_price
    FROM cohort_product_packing_units cpu
    JOIN packing_unit_products pu ON pu.id = cpu.product_packing_unit_id
    WHERE cpu.cohort_id IN ({','.join(str(c) for c in COHORT_IDS)})
        AND cpu.created_at::DATE <> '2023-07-31'
        AND cpu.is_customized = TRUE
        and basic_unit_count = 1 
    GROUP BY 1,2,3,4

'''
df_prices = query_snowflake(PRICES_QUERY)
#df_prices = setup_environment_2.dwh_pg_query(PRICES_QUERY, columns=['cohort_id','product_id','packing_unit_id','basic_unit_count','current_price'])
print(f'  Prices: {len(df_prices)} rows')

print('Loading packing units...')
PU_QUERY = '''
WITH sales_check AS (
    SELECT DISTINCT
        pso.product_id,
        pso.packing_unit_id,
        SUM(pso.total_price) AS nmv
    FROM product_sales_order pso
    JOIN sales_orders so ON so.id = pso.sales_order_id
    WHERE so.created_at >= CURRENT_DATE - 60 
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
    GROUP BY 1, 2
)
SELECT product_id, packing_unit_id, basic_unit_count
FROM (
    SELECT *, MAX(nmv) OVER (PARTITION BY product_id, is_basic_unit) AS top_nmv
    FROM (
        SELECT 
            pup.product_id,
            pup.packing_unit_id,
            pup.basic_unit_count,
            pup.is_basic_unit,
            COUNT(DISTINCT CASE WHEN pup.basic_unit_count = 1 THEN pup.packing_unit_id END) 
                OVER (PARTITION BY pup.product_id) AS total_basic,
            COALESCE(sc.nmv, 0) AS nmv
        FROM packing_unit_products pup
        LEFT JOIN sales_check sc 
            ON sc.product_id = pup.product_id 
            AND sc.packing_unit_id = pup.packing_unit_id
        WHERE pup.deleted_at IS NULL
    )
)
WHERE nmv = top_nmv OR (top_nmv = 0 AND total_basic <= 1)
'''
df_pus = query_snowflake(PU_QUERY)
print(f'  Packing units: {len(df_pus)} rows')

print('Loading product info...')
PRODUCT_QUERY = '''
SELECT p.id AS product_id,
       CONCAT(p.name_ar, ' ', p.size, ' ', product_units.name_ar) AS sku,
       b.name_ar AS brand,
       c.name_ar AS cat
FROM products p
JOIN brands b ON b.id = p.brand_id
JOIN categories c ON c.id = p.category_id
JOIN product_units ON product_units.id = p.unit_id
'''
df_products = query_snowflake(PRODUCT_QUERY)
print(f'  Products: {len(df_products)} rows')

print('Loading target margins...')
MARGIN_QUERY = f'''
SELECT brand, cat, target_margin
FROM (
    SELECT b.name_ar AS brand, c.name_ar AS cat, cplan.margin AS target_margin,
           ROW_NUMBER() OVER (PARTITION BY b.name_ar, c.name_ar ORDER BY cplan.date DESC) AS rn
    FROM performance.commercial_targets cplan
    JOIN brands b ON b.id = cplan.brand_id
    JOIN categories c ON c.id = cplan.category_id
    WHERE cplan.margin IS NOT NULL
    QUALIFY CASE 
        WHEN DATE_TRUNC('month', MAX(date) OVER()) = DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE) 
        THEN DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
        ELSE DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - INTERVAL '1 month') 
    END = DATE_TRUNC('month', date)
)
WHERE rn = 1
'''
df_targets = query_snowflake(MARGIN_QUERY)
print(f'  Target margins: {len(df_targets)} rows')

print('Loading stocks...')
STOCKS_QUERY = '''
WITH parent_whs AS (
    SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
)
SELECT warehouse_id, product_id,
    CASE WHEN stocks_child IS NOT NULL AND stocks = 0 THEN stocks_child ELSE stocks END AS stocks
FROM (
    SELECT pw.warehouse_id, pw.product_id,
        pw.available_stock::INTEGER AS stocks,
        pw2.available_stock::INTEGER AS stocks_child
    FROM product_warehouse pw
    LEFT JOIN parent_whs p ON p.parent_id = pw.warehouse_id
    LEFT JOIN product_warehouse pw2 ON pw2.warehouse_id = p.child_id AND pw.product_id = pw2.product_id
    WHERE pw.warehouse_id NOT IN (6, 9, 10)
        AND pw.is_basic_unit = 1
)
'''
df_stocks = query_snowflake(STOCKS_QUERY)
print(f'  Stocks: {len(df_stocks)} rows')

print('\nAll data loaded.')

/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json
/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json
Queries Module | Timezone: America/Los_Angeles
✅ UTH and Last Hour functions defined
✅ Yesterday closing stock function defined
Fixed price & cart rule functions defined
get_commercial_min_prices() defined
get_commercial_price_ups() defined
get_margin_boundaries_region() defined
get_margin_boundaries_global() defined

QUERIES MODULE READY

Live Data Functions:
  • get_current_stocks()
  • get_packing_units()
  • get_current_prices()
  • get_current_wac()
  • get_current_cart_rules()

UTH Performance Functions:
  • get_uth_performance()         - UTH qty/retailers (Snowflake)
  • get_hourly_distribution()     - Historical hour contributions (Snowflake)
  • get_last_hour_performance()   - Last hour qty/retailers (DWH)

Stock Functions:
  • get_yesterday_closing_stock() - Yesterday closing stock with parent WH mapping

Override Functions:
  • get_fixed_prices()

In [9]:
# =============================================================================
# BUILD LOOKUP TABLE (product_id → market prices + WAC + target margin)
# =============================================================================

# V2 market_data returns (product_id, region, price_tiers, wac_p, target_margin, num_sources, market_data_source)
# Expand to cohorts for lookup
df_market_cohorts = expand_to_cohorts(market_data)

# Lookup is per (product_id, cohort_id)
lookup = df_products.merge(df_wac, on='product_id', how='left')

cohort_df = pd.DataFrame({'cohort_id': COHORT_IDS})
lookup = lookup.merge(cohort_df, how='cross')
lookup = lookup.merge(
    df_market_cohorts[['product_id', 'cohort_id', 'price_tiers', 'target_margin']],
    on=['product_id', 'cohort_id'], how='left'
)
lookup['price_tiers'] = lookup['price_tiers'].apply(lambda x: x if isinstance(x, list) else [])

# Fill target_margin for rows without V2 data
lookup = lookup.merge(df_targets, on=['brand', 'cat'], how='left', suffixes=('', '_tgt'))
for col in ['target_bm', 'target_bm_tgt', 'cat_target_margin', 'cat_target_margin_tgt']:
    if col in lookup.columns:
        lookup['target_margin'] = lookup['target_margin'].fillna(lookup[col])
lookup['target_margin'] = lookup['target_margin'].fillna(0.05)
lookup.drop(columns=[c for c in lookup.columns if c.startswith('target_bm') or c.startswith('cat_target')], inplace=True, errors='ignore')

# Derive market percentile prices from price_tiers list
def tiers_to_percentiles(tiers):
    if not tiers or len(tiers) == 0:
        return pd.Series({'market_min': np.nan, 'market_25': np.nan, 'market_50': np.nan,
                          'market_75': np.nan, 'market_max': np.nan, 'market_avg': np.nan})
    arr = np.array(tiers)
    return pd.Series({
        'market_min': arr[0],
        'market_25': np.percentile(arr, 25),
        'market_50': np.percentile(arr, 50),
        'market_75': np.percentile(arr, 75),
        'market_max': arr[-1],
        'market_avg': (arr[0] + arr[-1]) / 2,
    })

lookup = pd.concat([lookup, lookup['price_tiers'].apply(tiers_to_percentiles)], axis=1)

# Merge current base-unit price per (product_id, cohort_id)
base_prices = df_prices[df_prices['basic_unit_count'] == 1][['cohort_id', 'product_id', 'current_price']].drop_duplicates()
lookup = lookup.merge(base_prices, on=['product_id', 'cohort_id'], how='left')

# Merge stocks (sum across warehouses per product for display; warehouse-level for push filtering)
wh_df = pd.DataFrame(WAREHOUSE_MAPPING, columns=['region', 'warehouse', 'warehouse_id', 'cohort_id'])
product_stocks = df_stocks.groupby('product_id')['stocks'].sum().reset_index().rename(columns={'stocks': 'total_stocks'})
lookup = lookup.merge(product_stocks, on='product_id', how='left')
lookup['total_stocks'] = lookup['total_stocks'].fillna(0).astype(int)

print(f'Lookup table: {len(lookup)} rows (product x cohort)')
print(f'  With market data: {(lookup["price_tiers"].apply(len) > 0).sum()}')
print(f'  With WAC: {lookup["wac_p"].notna().sum()}')
print(f'  With target margin: {lookup["target_margin"].notna().sum()}')

Lookup table: 232879 rows (product x cohort)
  With market data: 44526
  With WAC: 76045
  With target margin: 232879


In [11]:
lookup[(lookup['total_stocks']>0)].to_excel('manual_data.xlsx')

In [ ]:
#lookup.to_excel('manual_data.xlsx')

In [ ]:
#lookup[(lookup['brand']=='حلو الشام')&(lookup['total_stocks']>0)]

In [14]:
# =============================================================================
# DEFINE YOUR SKU ACTIONS HERE
# =============================================================================
# Format: (product_id, action)
#   action can be: 'market_min', 'market_25', 'market_50', 'market_75',
#                  'market_max', 'market_avg', 'target_margin',
#                  'step_up', 'step_down', or a number (fixed price)
#
# step_up: next tier above current price in the price_tiers list
# step_down: next tier below current price in the price_tiers list
# Each product is automatically expanded to ALL cohorts.
# Market-based actions use the cohort-specific market price.
# Only SKUs with stock > 0 are pushed.

PUSH_LIST = [
(9787,700,450),
(9787,701,450),
(9787,704,462.4375),
(12870,701,212.625),
(7122,704,104.75),
(12653,703,143.125),
(11504,702,210),
(5650,701,545),
(5652,1123,545),
(82,702,505),
(82,703,505),
(82,704,505),
(82,1124,505),
(414,701,357),
(414,703,351),
(8923,700,351.5625),
(97,1123,61.5625),
(5736,701,599.75),
(83,700,568.75),
(83,701,578),
(83,703,545),
(83,1123,568.75),
(83,1124,568.75),
(955,701,260),
(6561,700,89),
(6565,701,259),
(3495,700,28),
(1814,1123,408),
(411,702,290),
(11290,700,308),
(11290,1123,317),
(3557,703,47.625),
(205,701,299),
(205,702,280),
(205,704,288),
(205,1126,288.125),
(5005,701,32.75),
(417,700,197.25),
(417,701,197.25),
(417,702,196.5),
(11936,1123,218.25),
(11936,1124,222.5),
(11936,1125,218.25),
(11936,1126,218.25),
(903,703,401.5),
(903,1124,398.9375),
(11489,701,278),
(11489,704,282),
(11914,701,395.75),
(6269,704,33.75),
(7566,701,65),
(12647,702,135.5),
(12647,703,135.75),
(7624,700,16.375),
(6259,701,'target_margin'),
(10940,700,218.25),
(10940,701,218.25),
(10940,1124,218.25),
(96,701,37.5),
(96,1123,38.125),
(5733,700,612),
(11288,703,308),
(11288,1124,308),
(361,700,197.5),
(361,701,205.5),
(361,702,196.5),
(361,703,197.5),
(361,704,197.5),
(21697,701,161.75),
(21712,701,14.25),
(84,701,445),
(84,703,453.25),
(84,1123,460.75),
(84,1124,450),
(5722,700,611),
(2805,703,'target_margin'),
(2805,1125,'target_margin'),
(7119,700,103),
(7119,701,106.5),
(10938,1123,218.25),
(12508,700,340),
(19932,703,76.5),
(9785,704,447.25),
(9785,1123,447.25),
(10717,701,275),
(10398,700,218.5),
(10398,701,218.5),
(10936,704,212.5),
(10936,1125,212.5),
(10936,1126,212.5),
(10937,701,222.5),
(10937,1124,212.5),
(10937,1125,212.5),
(10937,1126,212.5),
(418,701,210),
(418,702,210),
(418,1123,217.875),
(418,1125,210),
(7005,701,387.5),
(5656,1123,545),
(3475,703,520),
(958,701,273.125),
(958,702,273.125),
(958,704,265.25),
(93,700,64.5),
(523,703,201.75),
(523,1123,197.25),
(523,1124,201.9375),
(957,700,281.5),
(957,701,281.5),
(957,704,281.5),
(11160,1123,59.75),
(9777,701,717.25),
(9784,700,450),
(9786,700,463.125),
(9786,1123,463.125),
(7630,1125,197.5),
(7630,1126,225),
(7885,701,205),
(7885,1126,210.125),
(7887,700,213.75),
(10147,1125,457.5),
(12656,700,376.75),
(12993,701,12),
(12993,704,12),
(12993,1125,12),
(6339,700,109),
(950,700,510),
(950,701,511.5),
(950,704,511.5),
(11491,700,210),
(11491,701,217.875),
(11491,702,210),
(11491,704,217.5),
(9372,700,213.375),
(9372,701,208),
(9372,704,213.375),
(3559,704,51.125),
(248,701,568),
(248,703,550),
(3477,701,585),
(5719,701,523.75),
(956,701,281.5),
(956,704,265.25),
(956,1125,265.25),
(1523,701,903.25),
(415,700,347),
(415,702,347),
(415,703,351.25),
(13213,701,174.1875),
(60,704,36),
(60,1124,35),
(60,1126,35),
(13210,700,374),
(13210,701,410),
(7120,701,106.5),
(19501,702,197.5),
(6566,700,255.5),
(19980,701,525),
(151,701,444),
(151,703,444),
(151,1125,444),
(142,701,201.9375),
(142,704,197.25),
(142,1124,197.25),
(61,700,142.5),
(61,701,133),
(75,701,224.25),
(98,700,39),
(98,701,37.5),
(202,703,407.5625),
(202,1124,408),
(206,703,288.75),
(332,700,410.875),
(410,701,288.75),
(410,1125,278),
(413,701,357),
(413,702,347),
(413,703,350.75),
(413,1125,347),
(412,701,355),
(412,702,355),
(412,704,371.75),
(6630,700,304.8125),
(10939,701,218.25),
(10939,704,212.5),
(10939,1123,212.5),
(10939,1124,212.5),
(10939,1125,212.5),
(13481,701,165),
(19987,704,443.5),
(14041,701,51.25),
(19981,1123,455),
(358,701,680),
(10942,701,218.25),
(362,701,217.75),
(362,704,216.75),
(12992,701,12),
(270,704,104.5),
(7573,1123,83.3125),
(21793,702,141),
(22558,700,55),
(22558,701,55),
(22558,704,58),
(22558,1123,60.125),
(141,701,201.9375),
(141,702,199.75),
(141,704,197.5),
(22313,701,445),
(22364,704,946.25),
(22324,700,357.25),
(24378,700,159.25),
(22327,701,732.5),
(22327,1125,714.75),
(27012,1126,46.25),
(6220,700,110.25),
(6224,701,126.8125),
(6209,701,'target_margin'),
(6207,700,222.75),
(6221,700,117.0625),
(19931,701,79.25),
(19931,704,76.25)
]


# =============================================================================
# COMPUTE NEW PRICES (per product × cohort)
# =============================================================================
ACTION_TO_COLUMN = {
    'market_min': 'market_min',
    'market_25': 'market_25',
    'market_50': 'market_50',
    'market_75': 'market_75',
    'market_max': 'market_max',
    'market_avg': 'market_avg',
}

results = []
for product_id,cohort_id, action in PUSH_LIST:
    product_rows = lookup[(lookup['product_id'] == product_id)&(lookup['cohort_id'] == cohort_id)] #
    if product_rows.empty:
        results.append({'product_id': product_id, 'cohort_id': '-', 'action': str(action), 'status': 'NOT FOUND'})
        continue

    for _, row in product_rows.iterrows():
        cohort_id = int(row['cohort_id'])
        wac = row.get('wac_p', None)
        sku = row.get('sku', '')
        brand = row.get('brand', '')

        if isinstance(action, (int, float)):
            base_price = float(action)
            action_label = f'fixed_{action}'
        elif action == 'target_margin':
            tm = row.get('target_margin', 0.05)
            if pd.isna(wac) or wac <= 0:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO WAC'})
                continue
            base_price = wac / (1 - tm)
            action_label = f'target_margin ({tm:.1%})'
        elif action == 'step_up':
            tiers = row.get('price_tiers', [])
            cur = row.get('current_price', 0)
            if not tiers or pd.isna(cur) or cur <= 0:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO TIERS OR PRICE'})
                continue
            next_up = [t for t in tiers if t > cur + 0.25]
            if not next_up:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'ALREADY AT TOP'})
                continue
            base_price = next_up[0]
            action_label = f'step_up ({cur:.2f} -> {base_price:.2f})'
        elif action == 'step_down':
            tiers = row.get('price_tiers', [])
            cur = row.get('current_price', 0)
            if not tiers or pd.isna(cur) or cur <= 0:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO TIERS OR PRICE'})
                continue
            next_down = [t for t in reversed(tiers) if t < cur - 0.5]
            if not next_down:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'ALREADY AT BOTTOM'})
                continue
            base_price = next_down[0]
            action_label = f'step_down ({cur:.2f} -> {base_price:.2f})'
        elif action in ACTION_TO_COLUMN:
            col = ACTION_TO_COLUMN[action]
            val = row.get(col, None)
            if pd.isna(val):
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO MARKET DATA'})
                continue
            base_price = val
            action_label = action
        else:
            results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': str(action), 'status': 'INVALID ACTION'})
            continue

        base_price_rounded = np.round(base_price * 4) / 4
        margin = (base_price_rounded - wac) / base_price_rounded if (wac and wac > 0 and base_price_rounded > 0) else None

        cur_price = row.get('current_price', None)

        results.append({
            'product_id': product_id,
            'cohort_id': cohort_id,
            'sku': sku,
            'brand': brand,
            'action': action_label,
            'wac': wac,
            'current_price': cur_price,
            'base_price': base_price,
            'new_price': base_price_rounded,
            'margin': margin,
            'status': 'OK',
        })

df_results = pd.DataFrame(results)

ok_count = (df_results['status'] == 'OK').sum()
err_count = len(df_results) - ok_count
products_ok = df_results[df_results['status'] == 'OK']['product_id'].nunique()
print(f'Results: {ok_count} OK across {products_ok} products × cohorts, {err_count} errors/skipped')
if err_count > 0:
    print('\nErrors/Skipped:')
    print(df_results[df_results['status'] != 'OK'][['product_id', 'cohort_id', 'sku', 'action', 'status']].to_string(index=False))
df_results[df_results['status'] == 'OK']

Results: 222 OK across 126 products × cohorts, 0 errors/skipped


,product_id,cohort_id,sku,brand,action,wac,current_price,base_price,new_price,margin,status
0,9787,700,عصير جهينة بيور كوكتيل - 235 مل,جهينة عصاير,fixed_450,435.542526,464.25,450.000000,450.00,0.032128,OK
1,9787,701,عصير جهينة بيور كوكتيل - 235 مل,جهينة عصاير,fixed_450,435.542526,470.00,450.000000,450.00,0.032128,OK
2,9787,704,عصير جهينة بيور كوكتيل - 235 مل,جهينة عصاير,fixed_462.4375,435.542526,485.00,462.437500,462.50,0.058286,OK
3,12870,701,كلوروكس ابيض باوش - 500 جم,كلوركس,fixed_212.625,190.820000,223.00,212.625000,212.50,0.102024,OK
4,7122,704,كابتشينو بونجورنو موكا 12 ظرف - 14 جم,بونجورنو,fixed_104.75,99.043203,111.25,104.750000,104.75,0.054480,OK
...,...,...,...,...,...,...,...,...,...,...,...
217,6209,701,امريكانا برجر توابل شرقيه عرض 30 قطعة بدل 25 ...,أمريكانا,target_margin (5.0%),94.180298,150.50,99.137156,99.25,0.051080,OK
218,6207,700,برجر حلوانى بيف 20 قطعة - 1 كجم,حلوانى اخوان,fixed_222.75,211.662171,240.00,222.750000,222.75,0.049777,OK
219,6221,700,اجنحة كوكى كرانشى حار - 700 جم,كوكى,fixed_117.0625,108.478805,120.75,117.062500,117.00,0.072831,OK
220,19931,701,برسيل جل لافندر - 70 جم,برسيل,fixed_79.25,72.610001,83.00,79.250000,79.25,0.083785,OK


In [15]:
df_results=df_results[df_results['current_price']>df_results['new_price']]
df_results['change'] = (df_results['new_price'] - df_results['current_price'])/df_results['current_price']
print(df_results['margin'].mean())
print(df_results['margin'].median())
print(df_results['margin'].min())
print(df_results['margin'].max())
#df_results=df_results[~df_results['brand'].isin(['كوكا كولا','شويبس'])]
df_results

0.061231640465562315
0.05757087739423689
-0.05128730268849195
0.23648441032429357


,product_id,cohort_id,sku,brand,action,wac,current_price,base_price,new_price,margin,status,change
0,9787,700,عصير جهينة بيور كوكتيل - 235 مل,جهينة عصاير,fixed_450,435.542526,464.25,450.000000,450.00,0.032128,OK,-0.030695
1,9787,701,عصير جهينة بيور كوكتيل - 235 مل,جهينة عصاير,fixed_450,435.542526,470.00,450.000000,450.00,0.032128,OK,-0.042553
2,9787,704,عصير جهينة بيور كوكتيل - 235 مل,جهينة عصاير,fixed_462.4375,435.542526,485.00,462.437500,462.50,0.058286,OK,-0.046392
3,12870,701,كلوروكس ابيض باوش - 500 جم,كلوركس,fixed_212.625,190.820000,223.00,212.625000,212.50,0.102024,OK,-0.047085
4,7122,704,كابتشينو بونجورنو موكا 12 ظرف - 14 جم,بونجورنو,fixed_104.75,99.043203,111.25,104.750000,104.75,0.054480,OK,-0.058427
...,...,...,...,...,...,...,...,...,...,...,...,...
217,6209,701,امريكانا برجر توابل شرقيه عرض 30 قطعة بدل 25 ...,أمريكانا,target_margin (5.0%),94.180298,150.50,99.137156,99.25,0.051080,OK,-0.340532
218,6207,700,برجر حلوانى بيف 20 قطعة - 1 كجم,حلوانى اخوان,fixed_222.75,211.662171,240.00,222.750000,222.75,0.049777,OK,-0.071875
219,6221,700,اجنحة كوكى كرانشى حار - 700 جم,كوكى,fixed_117.0625,108.478805,120.75,117.062500,117.00,0.072831,OK,-0.031056
220,19931,701,برسيل جل لافندر - 70 جم,برسيل,fixed_79.25,72.610001,83.00,79.250000,79.25,0.083785,OK,-0.045181


In [ ]:
# mona_data = pd.read_excel('mona_data.xlsx')
# for cohort_id in mona_data.NEW_COHORT_ID.unique():
#     out = mona_data[mona_data['NEW_COHORT_ID'] == cohort_id]
#     id_ = cohort_id
#     name = out.NEW_COHORT_NAME.values[0]
#     file_name_ = 'uploads/1_new_{}.xlsx'.format(name).replace(' ','_')
#     out.to_excel(file_name_,index = False,engine = 'xlsxwriter')
#     time.sleep(3)
#     ################### Loop to avoid file limit ######################
#     # split file into chunks
#     print('Spliting file into chunks...')
#     if id_ == 61:
#         chunks = [out[i:i + 2000] for i in range(0, len(out), 2000)]
#     else:
#         chunks = [out[i:i + 4000] for i in range(0, len(out), 4000)]
#     print(f'len chunks = {len(chunks)}')
#     fileslist = []
#     for i, chunk in tqdm(enumerate(chunks), total=len(chunks)):
#         fileslist.append(f'manual/output_{id_}_chunk_{i + 1}.xlsx')
#         output_file_path = f'manual/output_{id_}_chunk_{i + 1}.xlsx'
#         chunk.to_excel(output_file_path, index=False, engine='xlsxwriter')
#     # Loop over chunks and upload
#     print('Uploading...')
#     for file in fileslist:
#         chunk = file.split('chunk_')[1].split('.xls')[0]
#         x = post_prices(id_, file)
#         # print(str(x.content))
#         if ('"success":true' in str(x.content).lower()):
#             print(f"Prices are upoladed successfuly Region: {name} ,cohort: {id_}, chunk: {chunk}")
#         else:
#             print(f"ERROR Region: {name} ,cohort: {id_}, chunk: {chunk}")
#             print(x.content)
#             final_status = False
#             break

In [16]:
# =============================================================================
# PUSH PRICES
# =============================================================================
MODE = 'live'  # Change to 'live' to actually push

df_ok = df_results[df_results['status'] == 'OK'].copy()

# Filter to only SKUs with stock
if len(df_ok) > 0:
    stock_by_product = df_stocks.groupby('product_id')['stocks'].sum().reset_index()
    df_ok = df_ok.merge(stock_by_product, on='product_id', how='left')
    no_stock = df_ok['stocks'].fillna(0) <= 0
    if no_stock.any():
        print(f'Skipping {no_stock.sum()} rows with no stock')
        df_ok = df_ok[~no_stock].copy()

if len(df_ok) == 0:
    print('No valid prices to push.')
else:
    wh_df = pd.DataFrame(WAREHOUSE_MAPPING, columns=['region', 'warehouse', 'warehouse_id', 'cohort_id'])

    push_rows = []
    for _, r in df_ok.iterrows():
        target_cohort = int(r['cohort_id'])
        cohort_whs = wh_df[wh_df['cohort_id'] == target_cohort]

        if cohort_whs.empty:
            cohort_whs = pd.DataFrame([{'warehouse_id': 1, 'cohort_id': target_cohort}])

        cur_price = r['current_price'] if pd.notna(r.get('current_price')) else 0

        for _, wh in cohort_whs.iterrows():
            push_rows.append({
                'product_id': int(r['product_id']),
                'sku': r['sku'],
                'new_price': r['new_price'],
                'warehouse_id': int(wh['warehouse_id']),
                'cohort_id': target_cohort,
                'stocks': 1,
                'current_price': cur_price,
            })

    df_push = pd.DataFrame(push_rows)

    n_products = df_ok['product_id'].nunique()
    n_cohorts = df_ok['cohort_id'].nunique()
    print(f'Pushing {n_products} products × {n_cohorts} cohorts = {len(df_push)} rows')
    print(f'Mode: {MODE}')
    print()

    result = push_prices(
        df_prices=df_push,
        pus=df_pus,
        source_module='manual_push',
        mode=MODE,
    )
    print(f'\nDone. Result: {result}')

Pushing 126 products × 9 cohorts = 336 rows
Mode: live


🚀 MODE: LIVE
   Files will be prepared AND uploaded to API
Loading disable_pu_visibility from Google Sheets...
  ✓ Loaded 88 products to disable min PU visibility

PUSH PRICES - Source: manual_push
Total received: 336
Price changes to push: 336
Skipped (no change): 0

Price changes summary:
  Increases: 0
  Decreases: 336

🔗 Mirrored prices to 6 main/general cohorts (+202 rows)
   Cohort 695 ← 700: 54 rows
   Cohort 61 ← 700: 54 rows
   Cohort 699 ← 702: 16 rows
   Cohort 697 ← 703: 21 rows
   Cohort 698 ← 704: 35 rows
   Cohort 696 ← 1123: 22 rows

📋 Prepared 474 packing unit prices (incl. main cohorts)

Processing cohort: 61
  Saved: uploads/manual_push_61.xlsx (54 rows)
  Split into 1 chunks (size: 2000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 126.29it/s]

  Uploading...


    ✓ Chunk 1 uploaded successfully

Processing cohort: 695
  Saved: uploads/manual_push_695.xlsx (54 rows)
  Split into 1 chunks (size: 4000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 129.74it/s]

...

  Uploading...


    ✓ Chunk 1 uploaded successfully

Processing cohort: 1126
  Saved: uploads/manual_push_1126.xlsx (10 rows)
  Split into 1 chunks (size: 4000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 205.32it/s]

  Uploading...


    ✓ Chunk 1 uploaded successfully

🚀 UPLOAD COMPLETE
Mode: live
Total prepared: 474
Total failed: 0
  Cleanup: removed 30 temporary .xlsx files from 2 folder(s)

Done. Result: {'total_received': 336, 'price_changes': 336, 'pushed': 474, 'failed': 0, 'skipped': 0, 'source_module': 'manual_push', 'timestamp': '2026-04-21 13:12:47', 'mode': 'live', 'skipped_cohorts': []}
